# Turn-Taking and Endpointing — companion notebook

Sweep endpointing thresholds against caller pause patterns and plot the
**cut-in / latency frontier**: the curve you are actually choosing a point on
when you set a silence threshold.

Runs on the standard library plus matplotlib. No API key, no audio files.

## 1. The caller model

Each utterance is a list of `(spoken_ms, following_pause_ms, text_so_far_is_complete)`.
The **last** pause is the real end of turn; earlier ones are hesitations.
Replace `CALLS` with pauses measured from your own transcripts.

In [1]:
Utterance = list[tuple[int, int, bool]]

CALLS: dict[str, Utterance] = {
    "fluent":     [(1400, 900, True)],
    "hesitant":   [(700, 480, False), (900, 900, True)],
    "very slow":  [(600, 620, False), (400, 700, False), (500, 900, True)],
    "code-mixed": [(1100, 520, False), (600, 900, True)],
    "one word":   [(300, 900, True)],
}

for name, u in CALLS.items():
    pauses = ", ".join(f"{p}ms" for _s, p, _c in u)
    print(f"{name:<12} {len(u)} pause(s): {pauses}")

fluent       1 pause(s): 900ms
hesitant     2 pause(s): 480ms, 900ms
very slow    3 pause(s): 620ms, 700ms, 900ms
code-mixed   2 pause(s): 520ms, 900ms
one word     1 pause(s): 900ms


## 2. Two endpointing policies

`fixed_timer` waits the same time for everyone. `semantic` waits a short time
when the transcript already looks like a finished sentence and a long time when
it does not.

In [2]:
def fixed_timer(utt, min_delay_ms):
    """Return (cut_ins, added_latency_ms)."""
    for i, (_spoken, pause, _complete) in enumerate(utt):
        if pause >= min_delay_ms:
            return (0, min_delay_ms) if i == len(utt) - 1 else (1, 0)
    return 0, min_delay_ms


def semantic(utt, short_ms, long_ms):
    for i, (_spoken, pause, complete) in enumerate(utt):
        need = short_ms if complete else long_ms
        if pause >= need:
            return (0, need) if i == len(utt) - 1 else (1, 0)
    return 0, long_ms


def evaluate(policy):
    """Total cut-ins and mean added latency over the callers that were NOT cut off."""
    results = [policy(u) for u in CALLS.values()]
    cuts = sum(r[0] for r in results)
    ok = [r[1] for r in results if r[0] == 0]
    return cuts, (sum(ok) / len(ok) if ok else 0.0)


print("fixed  350 ms ->", evaluate(lambda u: fixed_timer(u, 350)))
print("fixed 1000 ms ->", evaluate(lambda u: fixed_timer(u, 1000)))
print("semantic      ->", evaluate(lambda u: semantic(u, 200, 800)))

fixed  350 ms -> (3, 350.0)
fixed 1000 ms -> (0, 1000.0)
semantic      -> (0, 200.0)


## 3. The frontier

Sweep the fixed threshold finely and plot both error types against it. The
semantic policy is a single point that should sit **below and left of** the
whole fixed-timer curve — less latency *and* fewer cut-ins.

In [3]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

grid = list(range(100, 1301, 5))
cuts = [evaluate(lambda u, d=d: fixed_timer(u, d))[0] for d in grid]
lats = [evaluate(lambda u, d=d: fixed_timer(u, d))[1] for d in grid]
sem_cuts, sem_lat = evaluate(lambda u: semantic(u, 200, 800))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.step(grid, cuts, where="post", label="fixed timer")
ax1.scatter([sem_lat], [sem_cuts], marker="*", s=260, zorder=5,
            label="semantic (200/800)")
ax1.set_xlabel("added latency paid by every caller (ms)")
ax1.set_ylabel("callers talked over")
ax1.set_title("The frontier — lower-left is better")
ax1.legend()
ax1.grid(alpha=.3)

ax2.plot(grid, cuts, label="cut-ins")
ax2.set_xlabel("min_delay (ms)")
ax2.set_ylabel("cut-ins", color="tab:blue")
ax2.grid(alpha=.3)
ax2b = ax2.twinx()
ax2b.plot(grid, lats, color="tab:orange", label="added latency")
ax2b.set_ylabel("added latency (ms)", color="tab:orange")
ax2.set_title("Both error types vs one threshold")

plt.tight_layout()
plt.show()

first_safe = next(d for d, c in zip(grid, cuts) if c == 0)
print(f"lowest fixed threshold with zero cut-ins : {first_safe} ms")
print(f"semantic reaches zero cut-ins at         : {sem_lat:.0f} ms")
print(f"ratio                                    : "
      f"{first_safe / sem_lat:.1f}x less added latency")

lowest fixed threshold with zero cut-ins : 705 ms
semantic reaches zero cut-ins at         : 200 ms
ratio                                    : 3.5x less added latency


## 4. What a wrong completeness classifier costs

Section 8 of the article warns that a real completeness detector is wrong a few
percent of the time. Here is what that does: flip the flag with probability `p`
and watch cut-ins climb. This is the number to weigh against the latency win.

In [4]:
import random

def noisy_semantic(utt, short_ms, long_ms, p_wrong, rng):
    for i, (_spoken, pause, complete) in enumerate(utt):
        seen = (not complete) if rng.random() < p_wrong else complete
        need = short_ms if seen else long_ms
        if pause >= need:
            return (0, need) if i == len(utt) - 1 else (1, 0)
    return 0, long_ms


print(f"{'p_wrong':>8} {'mean cut-ins':>13} {'mean latency':>13}")
for p in (0.0, 0.02, 0.05, 0.10, 0.25):
    trials = 400
    cs, ls = [], []
    for t in range(trials):
        rng = random.Random(t)
        c, l = evaluate(lambda u: noisy_semantic(u, 200, 800, p, rng))
        cs.append(c)
        ls.append(l)
    print(f"{p:>8.2f} {sum(cs)/trials:>13.2f} {sum(ls)/trials:>10.0f} ms")

print("\nA classifier this cheap only pays off while its error rate stays low;")
print("measure yours before trusting the frontier plot above.")

 p_wrong  mean cut-ins  mean latency
    0.00          0.00        200 ms
    0.02          0.08        214 ms
    0.05          0.20        234 ms
    0.10          0.36        263 ms
    0.25          0.91        357 ms

A classifier this cheap only pays off while its error rate stays low;
measure yours before trusting the frontier plot above.


## 5. Your turn

1. Replace `CALLS` with pause lengths measured from your own recordings — the
   shape of the frontier is dominated by *your* hesitation distribution.
2. Add a caller who trails off and never finishes. Note that **no** policy
   scores well, which is the article's §8 boundary condition.
3. Add the round-trip delay you pay when no local VAD is configured (the clock
   starts late) and re-read the frontier.